# 00 · 설정과 시험 (Askos-V1-80M)

- **무엇을:** 이후 노트북 01~10이 쓸 Modal·HF 환경을 한 번에 점검합니다. 점검 대상은 패키지, 설정, Volume, HF 비공개 저장소, 그리고 L4 · L40S · H100의 torch · flash-attn · triton입니다.
- **얼마나:** 첫 이미지 빌드 10~20분과 점검 10분 남짓이 걸립니다. GPU는 각각 몇 분씩만 씁니다(H100 · L40S · L4 각 1회).
- **결과:** `runs/00_setup/SETUP.md`(판정과 실측 숫자)를 만들고, HF 비공개 저장소 `<사용자>/askos-v1-80m-state`의 `00_setup/`에 같은 내용을 올립니다.

> 이 노트북은 학습을 하지 않습니다. `launch.sh`(deploy + spawn 1회)와 `status.sh`만 부릅니다. 셀이나 커널이 끊겨도 작업은 Modal에서 계속됩니다.

In [ ]:
!pip install -q "modal>=1.0" pyyaml numpy

## 사람이 터미널에서 직접 할 일 (한 번만)

```bash
modal setup                                              # 본인 Modal 계정 로그인
modal secret create huggingface HF_TOKEN=<HF 쓰기 토큰>   # 본인이 직접 타이핑
```

**토큰을 이 노트북에 붙여 넣지 마세요.** 코드는 Modal 시크릿이 넣어 주는 환경 변수로만 읽습니다.

## 1. 로컬 시험 (GPU·Modal·네트워크 없이)

In [ ]:
!python3 tests/test_infra.py && python3 -m py_compile modal_app.py ops.py common/*.py setup_check/*.py && bash -n launch.sh status.sh && echo "로컬 시험 통과"

## 2. 실행 (한 번만)

`CALL_ID.setup`이 이미 있으면 다시 띄우지 않습니다.

In [ ]:
!bash launch.sh setup

## 3. 상태 확인

15~20분마다 다시 실행합니다. `RUNNING`이면 기다리고, `FINISHED`면 결과가 `runs/00_setup/`에 저장됩니다.

In [ ]:
!bash status.sh setup

## 4. 결과 요약

In [ ]:
import json, pathlib
out = pathlib.Path("runs/00_setup")
s = json.loads((out / "summary.json").read_text())
print("판정:", s["verdict"], "| 상태 저장소:", s.get("state_repo"))
for name, rep in (s.get("parts") or {}).items():
    if not rep or "checks" not in rep:
        print(f"- {name}: 결과 없음 {rep}"); continue
    print(f"- {name}: {'통과' if rep['verdict']['pass'] else '실패'}  실패={rep['verdict']['failed']}  경고={rep['verdict']['warnings']}")
    for c in rep["checks"]:
        d = c["detail"]
        keys = [k for k in ("tflops_bf16", "max_abs_err", "allocated_gb", "write_mb_s", "read_mb_s", "mb_s", "ram_gb", "cpus_visible", "name") if k in d]
        if keys: print("    ", c["name"], {k: d[k] for k in keys})
print("\n전체 표:", out / "SETUP.md")

## 다음

- 판정이 **통과**면 `01_data_download.ipynb`로 넘어갑니다.
- **실패**면 `runs/00_setup/SETUP.md`의 실패 항목을 `AGENT_INSTRUCTIONS.md`의 문제 대처표와 맞춰 봅니다. 재실행은 사람이 승인한 뒤 `bash launch.sh setup --again`으로 합니다.
- 실행이 끝난 뒤 에이전트에게 준 Modal · HF 토큰은 폐기하세요.